<div style="background:linear-gradient(135deg,#123524 0%,#1f5a45 45%,#2B5A82 100%);color:#fff;padding:40px 36px;border-radius:16px;font-family:system-ui,-apple-system,'Segoe UI',sans-serif">
  <div style="font-size:12px;letter-spacing:.16em;opacity:.75;text-transform:uppercase">43008 Reinforcement Learning &middot; Assignment 3 &middot; Team LMF</div>
  <div style="font-size:46px;font-weight:800;line-height:1.05;margin:12px 0 6px">Experiment: REINFORCE</div>
  <div style="font-size:19px;opacity:.92">Monte-Carlo Policy Gradient on AgriFlight-RL, crop-row flight stabilisation</div>
  <div style="height:1px;background:rgba(255,255,255,.28);margin:24px 0 18px"></div>
  <div style="display:flex;flex-wrap:wrap;gap:36px">
    <div><div style="font-size:11px;letter-spacing:.12em;opacity:.7;text-transform:uppercase">Student</div><div style="font-size:20px;font-weight:600">Juan David Rodriguez Arciniegas</div></div>
    <div><div style="font-size:11px;letter-spacing:.12em;opacity:.7;text-transform:uppercase">Student ID</div><div style="font-size:20px;font-weight:600">25418551</div></div>
    <div><div style="font-size:11px;letter-spacing:.12em;opacity:.7;text-transform:uppercase">Environment</div><div style="font-size:20px;font-weight:600">AgroDrone-v0 &middot; MuJoCo Skydio X2</div></div>
  </div>
</div>

### Contents

1. [Introduction](#intro): question, hypothesis, success criterion
2. [Setup](#setup): environment, config, seeds, shared helpers
3. [Theory](#theory): the algorithm in your own words
4. [Implementation](#implementation): the code
5. [Training](#training): run, curve, stability
6. [Evaluation](#evaluation): held-out seeds, baselines, difficulty levels
7. [Results](#results): table and plots
8. [Discussion](#discussion): what worked, what did not, limits
9. [Contribution and reproducibility](#contribution): who did what, versions, references

<div style="border-left:4px solid rgb(140,90,43);background:rgba(140,90,43,0.10);padding:10px 14px;border-radius:0 8px 8px 0;margin:6px 0"><b>Template status (delete this box before submitting).</b> <code>Run all</code> works today and produces the baseline numbers. Cells marked <i>Write here</i> or <code>TODO</code> are yours. Do not edit the cells labelled <i>Shared, identical in all three notebooks</i>: the comparison is only fair if they stay the same.</div>

<a id="intro"></a>

## 1. Introduction

<div style="border-left:4px solid rgb(74,122,94);background:rgba(74,122,94,0.10);padding:8px 12px;border-radius:0 8px 8px 0;margin:6px 0"><b>Draft in plain language.</b> Read it, make sure you can explain every sentence out loud, then rewrite it in your own voice.</div>

**Research question.** Can REINFORCE, the simplest policy-gradient method, learn to steer the drone along the crop row in wind, and how far up the difficulty ladder (L0 to L4) does it get compared with A2C and PPO?

**What REINFORCE is, in two sentences.** REINFORCE flies one whole episode, adds up the rewards, and then makes the commands it used more likely if the episode went well and less likely if it went badly. It learns only from complete flights and has no critic, so it is the simplest and noisiest of the three methods.

**Hypothesis** (Part B expects this). REINFORCE will learn the easy levels but be noisy. Its learning signal is the raw total score of an episode, so one lucky gust or one unlucky start can mislead it, and training curves will differ a lot between seeds. I expect it to clear L0, probably L1, and to struggle from L2 (wind) upwards.

**Success criterion (written before seeing any results).** *Draft: agree the exact thresholds with the team, so all three algorithms use the same definition of "clearing" a level.*
- A level is **cleared** on the 20 held-out seeds when the crash rate is at most 10%. At L0 the mean altitude error must also be at most 5 cm. From L1 up, the drone must finish the row in at least 80% of episodes.
- This notebook succeeds if L0 is cleared with all 3 training seeds and L1 with at least 2 of 3. Anything higher is a bonus, and failing at L2 or above is an acceptable, reportable result.

<a id="setup"></a>

## 2. Setup

**2.1 Bootstrap.** Finds the repo (or clones it), fetches the Skydio X2 model from the public MuJoCo Menagerie and installs any missing dependencies. Works on **AWS SageMaker**, **Google Colab** and a local Jupyter. Nothing to edit.

<details>
<summary><b>First time on SageMaker? Click here: GitHub token + clone the repo (one-off, about 5 minutes)</b></summary>

The repo is **private**, so SageMaker needs a GitHub *personal access token* to clone it. Each team member creates their own.

**A. Create a fine-grained token on GitHub**
1. GitHub (top-right avatar) → **Settings** → **Developer settings** → **Personal access tokens** → **Fine-grained tokens** → **Generate new token**.
2. *Token name*: `sagemaker-agroflight`. *Expiration*: 30 days (covers the project).
3. *Resource owner*: `Nelkit`. *Repository access*: **Only select repositories** → `agroflight-rl`. (If `Nelkit` is not in the list, ask Nelkit to confirm you are a collaborator; a classic token with the `repo` scope also works.)
4. *Permissions* → *Repository permissions* → **Contents: Read and write** (write lets you push your results).
5. **Generate token** and copy it (`github_pat_...`). GitHub shows it only once. Never paste it into a notebook cell or commit it.

**B. Clone the repo in SageMaker (from a Terminal, not from a notebook cell)**

In SageMaker Studio / JupyterLab: *File → New → Terminal*, then:
```bash
cd ~
git clone https://<YOUR_TOKEN>@github.com/Nelkit/agroflight-rl.git
cd agroflight-rl
git remote set-url origin https://github.com/Nelkit/agroflight-rl.git   # keep the token out of .git/config
git config user.name  "Your Name"
git config user.email "your.email@student.uts.edu.au"
git config credential.helper store                                     # asks for the token once on the next pull/push
git checkout -b <yourname>/<topic>                                       # one branch per person, e.g. nelkit/ppo-l0
```
When `git pull`/`git push` asks: *Username* = your GitHub user, *Password* = the token.

**C. Run**
1. Open `~/agroflight-rl/notebooks/<this notebook>.ipynb` and pick a **PyTorch** kernel (CPU is fine for Stage 1).
2. Run this cell. It detects the clone, so no token is needed here.
3. Get the latest code at any time with `git pull` in the Terminal. Commit `results/*.json` and your notebook on your branch; checkpoints (`*.pt`) are git-ignored and stay in your SageMaker home.

**Colab or a notebook opened outside the clone:** this cell clones the repo itself. It reads the token from the `GITHUB_TOKEN` environment variable, from Colab *Secrets* (key icon → add `GITHUB_TOKEN`), or asks for it with a hidden prompt. On a local machine it never prompts: if the notebook is not inside a clone, the cell stops with a message saying what to do. The token is not saved in the notebook or in `.git/config`.

Full guide: `docs/sagemaker_setup.md`.
</details>

In [ ]:
import os, sys, subprocess, time, importlib, importlib.util
NOTEBOOK_T0 = time.time()

IN_COLAB = "google.colab" in sys.modules
IN_SAGEMAKER = bool(
    os.environ.get("SAGEMAKER_INTERNAL_IMAGE_URI") or os.environ.get("SM_CURRENT_HOST")
    or os.path.isdir("/opt/ml/metadata") or os.path.expanduser("~") == "/home/sagemaker-user"
)
PLATFORM = "colab" if IN_COLAB else ("sagemaker" if IN_SAGEMAKER else "local")

REPO_URL = "github.com/Nelkit/agroflight-rl.git"   # private repo: needs a GitHub token (see 2.1)
BRANCH = "main"                                    # only used if this cell has to clone the repo itself
CLONE_DIR = "/content/agroflight-rl" if IN_COLAB else os.path.expanduser("~/agroflight-rl")

def run(cmd, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True)

def find_repo_root(start):
    p = os.path.abspath(start)
    while True:
        if os.path.isfile(os.path.join(p, "pyproject.toml")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            return None
        p = parent

# 1. Repo: this notebook's own clone, an existing clone in CLONE_DIR, or clone it now.
#    Search upwards from the notebook file (VS Code exposes its path) and from the kernel's
#    working directory, so a kernel started outside the repo still finds it.
_nb_file = globals().get("__vsc_ipynb_file__")   # set by VS Code, None elsewhere
_starts = ([os.path.dirname(_nb_file)] if _nb_file else []) + [os.getcwd()]
REPO = next((r for r in map(find_repo_root, _starts) if r), None)
if REPO is None and os.path.isfile(os.path.join(CLONE_DIR, "pyproject.toml")):
    REPO = CLONE_DIR
if REPO is None:
    token = os.environ.get("GITHUB_TOKEN")
    if not token and IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token and PLATFORM == "local":
        # Never prompt on a local machine: in VS Code the prompt is easy to miss and the cell looks hung.
        raise RuntimeError(
            f"agroflight-rl repo not found from {os.getcwd()}. Open this notebook from inside your "
            f"clone and restart the kernel, or set GITHUB_TOKEN to let this cell clone it into {CLONE_DIR}.")
    if not token:
        from getpass import getpass
        token = getpass("GitHub token for the private repo (hidden, not stored): ").strip()
    if not token:
        raise RuntimeError("No GitHub token given, so the private repo cannot be cloned. See 2.1.")
    # GIT_TERMINAL_PROMPT=0: a wrong token fails straight away instead of waiting for a password.
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, f"https://{token}@{REPO_URL}", CLONE_DIR],
                   check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    run(["git", "remote", "set-url", "origin", "https://" + REPO_URL], cwd=CLONE_DIR)  # drop the token
    del token
    REPO = CLONE_DIR
os.chdir(REPO)

# 2. Drone model: Skydio X2 from the public google-deepmind/mujoco_menagerie (sparse clone, a few MB).
DRONE_XML = os.path.join(REPO, "mujoco_menagerie", "skydio_x2", "x2.xml")
if not os.path.isfile(DRONE_XML):
    if not os.path.isdir("mujoco_menagerie"):
        run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
             "https://github.com/google-deepmind/mujoco_menagerie.git", "mujoco_menagerie"])
    run(["git", "sparse-checkout", "set", "skydio_x2"], cwd="mujoco_menagerie")
os.environ["AGRO_DRONE_XML"] = DRONE_XML

# 3. Dependencies: install the project (+ torch, matplotlib, pandas) only if something is missing.
if any(importlib.util.find_spec(m) is None for m in ("mujoco", "gymnasium", "torch", "matplotlib", "pandas")):
    run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"])
    importlib.invalidate_caches()
sys.path.insert(0, os.path.join(REPO, "src"))
import mujoco, gymnasium   # also used by the reproducibility cell in Section 9

_head = subprocess.run(["git", "log", "-1", "--format=%h (%ad)", "--date=short"], cwd=REPO,
                       capture_output=True, text=True).stdout.strip()
_branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=REPO,
                         capture_output=True, text=True).stdout.strip()
print("platform:", PLATFORM)
print("repo    :", REPO, "| branch", _branch, "| commit", _head)
print("model   :", DRONE_XML)

**2.2 Experiment config and seeds.** Training seeds start at 0. Evaluation seeds start at 1000, so they are held out from training.

In [ ]:
import random
import numpy as np
import torch

EXPERIMENT = dict(
    algorithm="REINFORCE",
    student_name="Juan David Rodriguez Arciniegas",
    student_id="25418551",
    env_id="AgroDrone-v0",
    train_seed=0,
    eval_seeds=list(range(1000, 1020)),
    level=0,   # start at L0; move up only when it works
)
BACKUP_TO_DRIVE = False   # set True in Colab to copy results and checkpoints to Google Drive (survives the runtime)

HPARAMS = dict(
    total_timesteps=200_000,   # starting guess, not tuned
    learning_rate=3e-4,
    gamma=0.99,
    normalize_returns=True,
    ent_coef=0.0,
    hidden=(64, 64),
)

random.seed(EXPERIMENT["train_seed"])
np.random.seed(EXPERIMENT["train_seed"])
torch.manual_seed(EXPERIMENT["train_seed"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

**2.3 Shared, identical in all three notebooks. Do not edit.** Results file schema (so one comparison notebook can read all three), checkpoint save and load (the viewer loads these), Drive backup, and the learning-curve plot.

In [ ]:
import json, datetime, subprocess, shutil
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

RESULTS_DIR = os.path.join(REPO, "results")
CHECKPOINT_DIR = os.path.join(REPO, "checkpoints")

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True).strip()
    except Exception:
        return None

def new_results():
    return dict(
        schema=1,
        algorithm=EXPERIMENT["algorithm"],
        student=dict(name=EXPERIMENT["student_name"], id=EXPERIMENT["student_id"]),
        env_id=EXPERIMENT["env_id"],
        git_commit=git_commit(),
        created=datetime.datetime.now().isoformat(timespec="seconds"),
        hparams=HPARAMS,
        training=dict(train_seed=EXPERIMENT["train_seed"], total_timesteps=None, wall_clock_s=None,
                      best_checkpoint=None, curve=[]),   # curve: list of {"timesteps": int, "mean_return": float}
        evaluation=dict(eval_seeds=EXPERIMENT["eval_seeds"], baselines={}, levels={}),
    )

def save_results(results):
    os.makedirs(RESULTS_DIR, exist_ok=True)
    path = os.path.join(RESULTS_DIR, f"{EXPERIMENT['algorithm'].lower()}_{EXPERIMENT['student_id']}.json")
    with open(path, "w") as f:
        json.dump(results, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    return path

def save_checkpoint(model, obs_dim, act_dim, tag="best"):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    path = os.path.join(CHECKPOINT_DIR, f"{EXPERIMENT['algorithm'].lower()}_{EXPERIMENT['student_id']}_{tag}.pt")
    torch.save(dict(algorithm=EXPERIMENT["algorithm"], student_id=EXPERIMENT["student_id"], hparams=HPARAMS,
                    obs_dim=obs_dim, act_dim=act_dim, state_dict=model.state_dict()), path)
    return path

def load_checkpoint(model, tag="best"):
    path = os.path.join(CHECKPOINT_DIR, f"{EXPERIMENT['algorithm'].lower()}_{EXPERIMENT['student_id']}_{tag}.pt")
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["state_dict"])
    return ckpt

def backup_to_drive(dirs):
    if not IN_COLAB:
        print("Not in Colab: nothing to back up.")
        return
    from google.colab import drive
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/agroflight"
    os.makedirs(dest, exist_ok=True)
    for d in dirs:
        if os.path.isdir(d):
            shutil.copytree(d, os.path.join(dest, os.path.basename(d)), dirs_exist_ok=True)
    print("backed up to", dest)

def plot_learning_curve(curve, window=10):
    xs = [p["timesteps"] for p in curve]
    ys = np.array([p["mean_return"] for p in curve], dtype=float)
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.plot(xs, ys, alpha=0.35, label="mean return")
    if len(ys) >= window:
        smooth = np.convolve(ys, np.ones(window) / window, mode="valid")
        ax.plot(xs[window - 1:], smooth, linewidth=2, label=f"moving average ({window})")
    ax.set_xlabel("environment steps")
    ax.set_ylabel("episode return")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()

RESULTS = new_results()

**2.4 Environment check.** One reset and one random step, to confirm the model loads.

In [ ]:
import gymnasium as gym
import agroflight.envs
from agroflight.envs import ENV_ID

env = gym.make(ENV_ID, level=EXPERIMENT["level"])
obs, _ = env.reset(seed=EXPERIMENT["train_seed"])
print("observation:", env.observation_space.shape, "| action:", env.action_space.shape)
obs, r, terminated, truncated, info = env.step(env.action_space.sample())
print("one random step -> reward", round(r, 3), "| crashed:", info["crashed"])
env.close()

<a id="theory"></a>

## 3. Theory

<div style="border-left:4px solid rgb(74,122,94);background:rgba(74,122,94,0.10);padding:8px 12px;border-radius:0 8px 8px 0;margin:6px 0"><b>Draft in plain language.</b> Read it, make sure you can explain every sentence out loud, then rewrite it in your own voice.</div>

#### 3.1 The idea in plain words

Imagine a coach who can only watch a whole run and then give one score. The player replays the run in their head: "in the runs that scored well I should do those things more often, and in the runs that scored badly, less often." That is REINFORCE. The drone flies a full episode with some randomness in its commands, we add up the rewards, and then we nudge the network so the commands that came before a good result become more likely, and the ones before a bad result become less likely.

#### 3.2 The steps

1. **Fly** one episode. At every step the network gives an *average* command, we add noise (sampling), and we remember how likely that exact command was (its log-probability).
2. **Score.** When the episode ends, compute for every step $t$ the *return* $G_t$: the discounted total reward from step $t$ to the end.
3. **Learn.** For every step, push the network's weights so that command becomes *more likely* if $G_t$ is high and *less likely* if $G_t$ is low, in proportion to $G_t$.
4. **Repeat** with the improved network.

Core objective, as a starting point to explain (not a substitute for the explanation):

$$\nabla_\theta J(\theta)=\mathbb{E}_{\tau\sim\pi_\theta}\Big[\sum_{t}\nabla_\theta\log\pi_\theta(a_t\mid s_t)\,G_t\Big],\qquad G_t=\sum_{k\ge t}\gamma^{\,k-t}r_k$$

#### 3.3 The equation, piece by piece

| Symbol | Plain meaning |
|---|---|
| $\theta$ | all the network's weights (what we are learning) |
| $\pi_\theta(a_t \mid s_t)$ | the probability the network gives to command $a_t$ when it senses $s_t$ |
| $\log \pi_\theta(a_t \mid s_t)$ | that probability on a log scale; its gradient says which way to change $\theta$ to make this command more likely |
| $G_t$ | how well things went from step $t$ onwards (the return) |
| $\nabla_\theta J$ | the direction in which the expected total score goes up |
| $\mathbb{E}$ | the average over many flights |

In words: **change the weights in the direction that makes each command more likely, weighted by how good the outcome was afterwards.** Good outcomes pull harder, and bad outcomes push the other way.

**Why the log trick?** The reward comes out of the simulator, and we cannot differentiate through the simulator. The log-derivative trick rewrites the gradient so that only the *network's own probabilities* need to be differentiated, while the environment only has to be *sampled*. That is what makes this a model-free method: we never use the simulator's equations.

#### 3.4 The return $G_t$, with a tiny example

$$G_t=r_t+\gamma\,r_{t+1}+\gamma^2 r_{t+2}+\dots$$

$\gamma$ (0.99 here) makes near rewards count more than far ones. Worked example with $\gamma=0.9$ and three rewards $[1,\,0,\,2]$:

- $G_2 = 2$
- $G_1 = 0 + 0.9\cdot 2 = 1.8$
- $G_0 = 1 + 0.9\cdot 1.8 = 2.62$

You compute it backwards, from the end of the episode: each return is the reward now plus $\gamma$ times the next return.

#### 3.5 Unbiased, but noisy

The push uses the *actual* return of one sampled episode. On average it points the right way (it is **unbiased**), but any single episode is a noisy estimate: one lucky gust makes a bad command look good, one unlucky start makes a good command look bad. That is the **high variance** Part B expects, and it is why REINFORCE needs many episodes and can look jumpy between seeds. A2C and PPO exist largely to reduce this noise.

**Cheap variance reduction (optional):** *return normalisation*, subtracting the mean of the returns and dividing by their standard deviation, so the push does not depend on the raw scale of the reward. It is a standard trick that usually helps a lot in practice. Subtracting a baseline from the returns is the same family of idea, and a *learned* baseline is what A2C's critic adds.

#### 3.6 A Gaussian policy for continuous actions

Our 4 commands are continuous, so the network cannot pick from a menu. Instead it outputs the **mean** $\mu_\theta(s)$ of each command, and there is a learnable **spread** $\sigma$ per command (stored as $\log\sigma$ so it stays positive). A command is sampled as $a=\mu_\theta(s)+\sigma\cdot\varepsilon$ with $\varepsilon\sim\mathcal N(0,1)$. That randomness is the **exploration**. The log-probability of the sampled 4-vector is the sum of the four per-command log-probabilities. At evaluation we drop the noise and use $a=\mu_\theta(s)$. The environment clips $a$ to $[-1,1]$, and we still compute the log-probability of the *unclipped* sample.

#### 3.7 Why the update waits for the end of the episode

$G_t$ needs the rewards that come *after* step $t$, so it cannot be computed until the episode is over. REINFORCE is therefore an *episodic, Monte-Carlo* method. A2C and PPO relax this by estimating the future with a critic.

#### 3.8 Pseudocode

```text
initialise policy weights θ and log-std
repeat for each episode:
    s ← env.reset()
    while not done:
        a ~ Normal(μ_θ(s), σ)            # sample a command
        store log π_θ(a | s)
        s, r, done ← env.step(a)
        store r
    compute G_t for every step (backwards)
    optionally normalise the G_t
    loss ← − Σ_t  log π_θ(a_t | s_t) · G_t
    θ ← θ − learning_rate · ∇θ loss      # one gradient step
```

<a id="implementation"></a>

## 4. Implementation

Write REINFORCE directly in PyTorch, following the subject's policy-gradient content. Keep the Gaussian policy head the same shape as the other two notebooks so the comparison stays fair.

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Normal

class GaussianPolicy(nn.Module):
    """obs -> Gaussian over the 4 action dimensions (mean network + learnable log-std)."""
    def __init__(self, obs_dim, act_dim, hidden=HPARAMS["hidden"]):
        super().__init__()
        # TODO: mean network (MLP) and a learnable log_std parameter

    def forward(self, obs):
        raise NotImplementedError  # TODO: return a torch.distributions.Normal

def compute_returns(rewards, gamma):
    raise NotImplementedError  # TODO: discounted return G_t for every step of one episode

def reinforce_update(policy, optimizer, log_probs, returns):
    raise NotImplementedError  # TODO: policy-gradient loss and one optimiser step

<a id="training"></a>

## 5. Training

<div style="border-left:4px solid rgb(74,122,94);background:rgba(74,122,94,0.10);padding:8px 12px;border-radius:0 8px 8px 0;margin:6px 0"><b>Draft in plain language.</b> Read it, make sure you can explain every sentence out loud, then rewrite it in your own voice.</div>

**Planned setup** (update this with what you actually ran):
- **Level:** start at L0, and move up one level only when the previous one is cleared.
- **Hyperparameters** (first guesses, set in the config cell): learning rate 3e-4, discount $\gamma=0.99$ (rewards about 100 steps, or 1 s, ahead still count), network 64 x 64, return normalisation on.
- **Seeds:** 3 training seeds (0, 1, 2), so I can show the spread and not just the best run.
- **Budget:** about 200,000 steps per run as a first pass, which is about 200 episodes of 1000 steps.
- **Measure:** wall-clock time per 1M steps (Part B asks Stage 1 to calibrate this).
- **Watch for:** a flat curve (suspect a bug or a learning rate that is too small) and a curve that jumps up and down (expected: this is the variance from Section 3.5).

In [ ]:
# TODO: training loop for REINFORCE.
# After every N steps append a point to the curve, for example:
# RESULTS["training"]["curve"].append({"timesteps": global_step, "mean_return": mean_episode_return})
# When finished, set RESULTS["training"]["total_timesteps"] and RESULTS["training"]["wall_clock_s"].
#
# Keep the best model by TRAINING-time return, never by the held-out evaluation seeds:
# RESULTS["training"]["best_checkpoint"] = save_checkpoint(model, obs_dim=14, act_dim=4, tag="best")

In [ ]:
if RESULTS["training"]["curve"]:
    plot_learning_curve(RESULTS["training"]["curve"])
else:
    print("No training run recorded yet.")

<a id="evaluation"></a>

## 6. Evaluation

Every policy is evaluated on the same held-out seeds (1000 to 1019) with exploration disabled. The two hand-coded baselines run at this notebook's level (`EXPERIMENT["level"]`, L0 by default); re-run them if the environment changes.

In [ ]:
from agroflight.evaluation.evaluate_policy import evaluate
from agroflight.baselines import altitude_hold_action, random_action

eval_env = gym.make(ENV_ID, level=EXPERIMENT["level"])
rng = np.random.default_rng(0)

RESULTS["evaluation"]["baselines"]["altitude_hold"] = evaluate(
    altitude_hold_action, seeds=EXPERIMENT["eval_seeds"], env=eval_env)
RESULTS["evaluation"]["baselines"]["random"] = evaluate(
    lambda obs: random_action(eval_env.action_space, rng), seeds=EXPERIMENT["eval_seeds"], env=eval_env)

for name, st in RESULTS["evaluation"]["baselines"].items():
    print(f"{name:<14} reward {st['mean_reward']:8.1f}  crash {st['crash_rate']:.0%}  finish {st['finish_rate']:.0%}")

In [ ]:
# TODO: after training, rebuild the model, call load_checkpoint(model, tag="best"), and set trained_policy
# to a function obs -> action (deterministic, no exploration).
trained_policy = None

# TODO: evaluate at every difficulty level this algorithm reaches (L0 to L4), creating one env per level,
# and store one entry per level, for example RESULTS["evaluation"]["levels"]["L0"] = evaluate(...)
if trained_policy is not None:
    RESULTS["evaluation"]["levels"]["L0"] = evaluate(trained_policy, seeds=EXPERIMENT["eval_seeds"], env=eval_env)
else:
    print("No trained REINFORCE policy yet: baselines only.")

<a id="results"></a>

## 7. Results

In [ ]:
import pandas as pd

rows = []
for name, st in RESULTS["evaluation"]["baselines"].items():
    rows.append({"policy": name, **st})
for level, st in RESULTS["evaluation"]["levels"].items():
    rows.append({"policy": f"REINFORCE @ {level}", **st})

cols = ["mean_reward", "std_reward", "crash_rate", "finish_rate", "mean_steps", "mean_abs_alt_error", "n_episodes"]
df = pd.DataFrame(rows).set_index("policy")[cols]
display(df.round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, col, title in zip(axes, ["mean_reward", "crash_rate", "finish_rate"],
                          ["Mean reward", "Crash rate", "Row completion rate"]):
    ax.bar(df.index, df[col])
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

path = save_results(RESULTS)
print("results saved to", path)
if BACKUP_TO_DRIVE:
    backup_to_drive([RESULTS_DIR, CHECKPOINT_DIR])

<div style="border-left:4px solid rgb(147,105,28);background:rgba(147,105,28,0.10);padding:10px 14px;border-radius:0 8px 8px 0;margin:6px 0"><b>Write here.</b> State the headline result in one sentence: the highest level cleared and the samples it needed. Compare against both baselines and against your success criterion from Section 1. Report what did not work as well as what did.</div>

<a id="discussion"></a>

## 8. Discussion

<div style="border-left:4px solid rgb(147,105,28);background:rgba(147,105,28,0.10);padding:10px 14px;border-radius:0 8px 8px 0;margin:6px 0"><b>Write here.</b> <ul><li>Was training stable across seeds? Show the spread, not just the best run.</li><li>Did the reward behave as intended, or did you see reward hacking (for example hovering in place)? Check the viewer, not only the number.</li><li>Where did the algorithm fail, and is that the algorithm, the reward, or the environment?</li><li>Limits: MuJoCo has no rotor ground effect or canopy turbulence; wind is observable; single environment.</li><li>What would you try next with more time?</li></ul></div>

<a id="contribution"></a>

## 9. Contribution and reproducibility

<div style="border-left:4px solid rgb(147,105,28);background:rgba(147,105,28,0.10);padding:10px 14px;border-radius:0 8px 8px 0;margin:6px 0"><b>Write here.</b> Contribution statement (marks are split by individual contribution). Fill in the table.</div>

| | |
|---|---|
| **My contribution** | *what Juan built, ran and wrote* |
| **Shared work with teammates** | *environment rework, evaluation harness, viewer, report sections* |
| **AI assistance used** | *Draft, edit so it is true:* Claude (Anthropic) explained RL concepts, scaffolded the notebook and the environment, and drafted the plain-language text in Sections 1, 3 and 5. **[what I checked, rewrote and understood myself]** |

In [ ]:
import platform
print("python    :", platform.python_version())
print("torch     :", torch.__version__)
print("numpy     :", np.__version__)
print("gymnasium :", gym.__version__)
print("mujoco    :", mujoco.__version__)
print("git commit:", git_commit())
print("train seed:", EXPERIMENT["train_seed"], "| eval seeds:", EXPERIMENT["eval_seeds"][0], "to", EXPERIMENT["eval_seeds"][-1])
print("notebook wall-clock so far: %.0f s" % (time.time() - NOTEBOOK_T0))

### References

1. Williams, R. J. (1992). Simple statistical gradient-following algorithms for connectionist reinforcement learning. Machine Learning 8, 229-256.
2. Sutton, R. S., McAllester, D., Singh, S., Mansour, Y. (2000). Policy gradient methods for reinforcement learning with function approximation. NeurIPS.
3. Todorov, E., Erez, T., Tassa, Y. (2012). MuJoCo: A physics engine for model-based control. IROS.
4. Google DeepMind. MuJoCo Menagerie: Skydio X2 model (Apache-2.0). github.com/google-deepmind/mujoco_menagerie.
5. Towers, M. et al. (2024). Gymnasium: A Standard Interface for Reinforcement Learning Environments. arXiv:2407.17032.